# Sesiunea 8 - Arbori de decizie si Random Forests

Azi clasificam cereri de credit: modelul trebuie sa prezica daca o cerere este **aprobata** sau **respinsa**.

Datasetul are mai multe coloane numerice si cateva coloane categorice simple. Este potrivit pentru arbori fiindca multe decizii pot fi gandite ca reguli cu praguri: scor de credit, venit, suma imprumutului, termen etc.

Pe parcurs vei avea exercitii scurte. Ruleaza celulele, modifica valorile propuse si noteaza ce observi.


## Pregatire (ruleaza o data)

Incarcam datasetul de credit dintr-un CSV public si pregatim coloanele pentru model.

Tinta este `loan_status`:

- `1` = Approved
- `0` = Rejected


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

plt.rcParams["figure.figsize"] = (8, 4.5)

URL = "https://raw.githubusercontent.com/fahadsultan/csc272/refs/heads/main/data/loan_approval_dataset.csv"

df_raw = pd.read_csv(URL)
df_raw.columns = [c.strip() for c in df_raw.columns]

df = df_raw.drop(columns=["loan_id"]).copy()

for col in ["education", "self_employed", "loan_status"]:
    df[col] = df[col].astype(str).str.strip()

df["education"] = (df["education"] == "Graduate").astype(int)
df["self_employed"] = (df["self_employed"] == "Yes").astype(int)
df["loan_status"] = (df["loan_status"] == "Approved").astype(int)

FEATURES = [c for c in df.columns if c != "loan_status"]
TARGET = "loan_status"
CLASS_NAMES = ["Respins", "Aprobat"]

df.head()


## 0. Datele: ce incercam sa prezicem?

Fiecare rand este o cerere de credit. Avem informatii despre solicitant, venit, suma ceruta, termen, scor de credit si valoarea unor active.

**Exercitiul 0.1 - Inspectie rapida**

Ruleaza celulele de mai jos si raspunde:

- cate randuri si coloane avem?
- clasele sunt echilibrate?
- care coloane par cele mai importante pentru decizia de credit?


In [ ]:
print("Dimensiune initiala:", df_raw.shape)
print("Dimensiune dupa pregatire:", df.shape)
print("Valori lipsa:", df.isna().sum().sum())
print()
print("Distributia tintei:")
print(df[TARGET].map({0: "Respins", 1: "Aprobat"}).value_counts())
print()
print("Proportii:")
print(df[TARGET].map({0: "Respins", 1: "Aprobat"}).value_counts(normalize=True).round(3))

df[["cibil_score", "income_annum", "loan_amount", "loan_term"]].describe().round(0)


In [ ]:
colors = df[TARGET].map({0: "tab:red", 1: "tab:green"})

plt.scatter(df["cibil_score"], df["loan_amount"], c=colors, alpha=0.35)
plt.xlabel("cibil_score")
plt.ylabel("loan_amount")
plt.title("Credit: scor de credit vs suma ceruta")
plt.show()


In [ ]:
for status, label, color in [(0, "Respins", "tab:red"), (1, "Aprobat", "tab:green")]:
    subset = df[df[TARGET] == status]
    plt.hist(subset["cibil_score"], bins=25, alpha=0.55, label=label, color=color)

plt.xlabel("cibil_score")
plt.ylabel("numar de cereri")
plt.title("Distributia scorului de credit")
plt.legend()
plt.show()


<details>
<summary><b>Observatie</b></summary>

`cibil_score` pare sa separe foarte bine cererile aprobate de cele respinse. Asta inseamna ca un model de tip arbore va putea gasi praguri utile pe aceasta coloana.
</details>


## 1. Baseline si split

Inainte de orice model, facem split train/test si calculam baseline-ul: ce obtinem daca alegem mereu clasa majoritara?

**Exercitiul 1.1 - Baseline**

Inainte sa rulezi, estimeaza acuratetea baseline-ului uitandu-te la distributia tintei.

Dupa rulare, compara orice model urmator cu acest baseline.


In [ ]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print(f"train: {len(X_train)} cereri | test: {len(X_test)} cereri")
print(f"Baseline accuracy: {accuracy_score(y_test, baseline_pred):.3f}")
print(f"Baseline F1:       {f1_score(y_test, baseline_pred):.3f}")


<details>
<summary><b>Observatie</b></summary>

Baseline-ul alege mereu clasa cea mai frecventa. Daca un model nu bate clar acest scor, nu a invatat un tipar util din date.
</details>


## 2. Fii tu modelul: scrie o regula manuala

Inainte sa antrenam un arbore, scriem noi o regula cu praguri.

**Exercitiul 2.1 - Regula ta**

Modifica pragurile din celula. Scopul este sa obtii o regula simpla si usor de explicat, nu neaparat perfecta.


In [ ]:
# Regula de pornire:
# - aprobam daca scorul de credit este suficient de mare
# - incearca apoi sa adaugi o conditie pe venit, suma sau termen

manual_pred = (X_test["cibil_score"] >= 600).astype(int)

print(f"Regula manuala accuracy: {accuracy_score(y_test, manual_pred):.3f}")
print(f"Regula manuala precision: {precision_score(y_test, manual_pred):.3f}")
print(f"Regula manuala recall:    {recall_score(y_test, manual_pred):.3f}")
print(f"Regula manuala F1:        {f1_score(y_test, manual_pred):.3f}")


In [ ]:
for prag in [450, 500, 550, 600, 650, 700]:
    pred = (X_test["cibil_score"] >= prag).astype(int)
    print(
        f"cibil_score >= {prag}: "
        f"accuracy={accuracy_score(y_test, pred):.3f} | "
        f"precision={precision_score(y_test, pred):.3f} | "
        f"recall={recall_score(y_test, pred):.3f}"
    )


<details>
<summary><b>Observatie</b></summary>

Un prag in jur de `550` pe `cibil_score` este deja foarte puternic. Asta pregateste ideea centrala: arborele va cauta automat astfel de praguri.
</details>


## 3. Primul arbore

Un arbore de decizie cauta intrebari de tipul: „coloana X este mai mica sau egala cu pragul P?”.

Incepem cu un arbore foarte mic: `max_depth=1`. Asta inseamna o singura intrebare.

**Exercitiul 3.1 - Ce intrebare alege arborele?**

Inainte sa rulezi, estimeaza ce coloana va alege arborele pentru prima intrebare.


In [ ]:
stump = DecisionTreeClassifier(max_depth=1, random_state=42)
stump.fit(X_train, y_train)

print(export_text(stump, feature_names=FEATURES))
print(f"Accuracy pe test: {stump.score(X_test, y_test):.3f}")


**Exercitiul 3.2 - Citeste arborele desenat**

Antrenam un arbore de adancime 3. Citeste-l de la radacina spre frunze si noteaza ce conditii duc la aprobare sau respingere.


In [ ]:
arbore = DecisionTreeClassifier(max_depth=3, random_state=42)
arbore.fit(X_train, y_train)

plt.figure(figsize=(16, 7))
plot_tree(
    arbore,
    feature_names=FEATURES,
    class_names=CLASS_NAMES,
    filled=True,
    impurity=False,
    fontsize=8,
)
plt.show()

print(f"Accuracy pe test: {arbore.score(X_test, y_test):.3f}")


<details>
<summary><b>Observatie</b></summary>

Arborele alege praguri pe coloanele care separa cel mai bine cererile aprobate de cele respinse. Un arbore mic poate fi citit aproape ca o regula scrisa de mana.
</details>


## 4. Cum alege arborele pragul? Gini

Arborele incearca mai multe taieturi si o prefera pe cea care lasa grupuri cat mai curate.

Pentru clasificare binara:

`Gini = 1 - p_aprobat² - p_respins²`

**Exercitiul 4.1 - Gini pentru doua clase**

Citeste functia si ruleaza exemplele.

Intrebare de verificare: cand este Gini 0 si cand devine mare?


In [ ]:
def gini_binar(etichete):
    if len(etichete) == 0:
        return 0.0
    p = np.mean(etichete)
    return 1 - p**2 - (1 - p)**2

print("grup pur aprobat:", gini_binar(np.array([1, 1, 1, 1])))
print("grup pur respins:", gini_binar(np.array([0, 0, 0, 0])))
print("grup 50/50:", gini_binar(np.array([1, 1, 0, 0])))
print("radacina train:", round(gini_binar(y_train), 3))


**Exercitiul 4.2 - Cautam manual pragul pe `cibil_score`**

Citeste codul si estimeaza unde va fi minimul. Apoi ruleaza si compara cu pragul ales de arbore.

Intrebare de verificare: de ce folosim media ponderata intre grupul din stanga si grupul din dreapta?


In [ ]:
rezultate = []
for prag in range(350, 900, 25):
    stanga = y_train[X_train["cibil_score"] < prag]
    dreapta = y_train[X_train["cibil_score"] >= prag]
    if len(stanga) == 0 or len(dreapta) == 0:
        continue

    gini_ponderat = (
        len(stanga) * gini_binar(stanga)
        + len(dreapta) * gini_binar(dreapta)
    ) / len(y_train)
    rezultate.append((prag, gini_ponderat))

rez = pd.DataFrame(rezultate, columns=["prag", "gini_ponderat"])

plt.plot(rez["prag"], rez["gini_ponderat"], marker="o")
plt.xlabel("prag pe cibil_score")
plt.ylabel("Gini ponderat")
plt.title("Unde ar taia arborele?")
plt.show()

rez.loc[rez["gini_ponderat"].idxmin()].round(3)


<details>
<summary><b>Observatie</b></summary>

Gini ponderat este mic atunci cand cele doua grupuri rezultate sunt cat mai curate. Pentru acest dataset, o taietura pe `cibil_score` separa foarte bine cererile aprobate de cele respinse.
</details>


## 5. Adancime, underfitting si overfitting

Un arbore prea mic poate rata reguli utile. Un arbore prea mare poate memora detalii din train.

**Exercitiul 5.1 - Train vs test**

Ruleaza mai multe valori pentru `max_depth`. Unde vezi underfitting? Unde poate aparea overfitting?


In [ ]:
randuri = []
for depth in range(1, 16):
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    randuri.append({
        "max_depth": depth,
        "train_accuracy": model.score(X_train, y_train),
        "test_accuracy": model.score(X_test, y_test),
        "frunze": model.get_n_leaves(),
    })

scoruri_depth = pd.DataFrame(randuri)

plt.plot(scoruri_depth["max_depth"], scoruri_depth["train_accuracy"], marker="o", label="train")
plt.plot(scoruri_depth["max_depth"], scoruri_depth["test_accuracy"], marker="o", label="test")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.legend()
plt.title("Cand crestem arborele, train-ul urca primul")
plt.show()

scoruri_depth.round(3)


<details>
<summary><b>Observatie</b></summary>

Daca scorul pe train creste mult, dar scorul pe test se plafoneaza sau scade, modelul incepe sa memoreze detalii din train. Pentru arbori, `max_depth` este una dintre cele mai simple frane.
</details>


## 6. Random Forest

Random Forest construieste multi arbori pe esantioane usor diferite si ii pune la vot. Rezultatul este de obicei mai stabil decat un singur arbore.

**Exercitiul 6.1 - Compara modele**

Comparam baseline, regresie logistica, un arbore mic si Random Forest.


In [ ]:
def metrici_model(nume, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    return {
        "model": nume,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC_AUC": roc_auc_score(y_test, proba),
    }

modele = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    "Regresie logistica": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "Arbore depth=3": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
}

rezultate_modele = pd.DataFrame([
    metrici_model(nume, model) for nume, model in modele.items()
]).set_index("model")

rezultate_modele.round(3)


**Exercitiul 6.2 - Matricea de confuzie**

Identifica ce tip de greseala face Random Forest:

- fals pozitiv: modelul aproba o cerere care era respinsa;
- fals negativ: modelul respinge o cerere care era aprobata.


In [ ]:
rf = modele["Random Forest"]
rf_pred = rf.predict(X_test)

cm = confusion_matrix(y_test, rf_pred)
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(cmap="Blues", colorbar=False)
plt.title("Random Forest - matrice de confuzie")
plt.show()

print(classification_report(y_test, rf_pred, target_names=CLASS_NAMES))

tn, fp, fn, tp = cm.ravel()
print(f"Aprobari corecte: {tp}")
print(f"Respingeri corecte: {tn}")
print(f"Aprobari gresite / fals pozitiv: {fp}")
print(f"Respingeri gresite / fals negativ: {fn}")


## 7. Ce coloane conteaza?

Random Forest ne da o aproximare a importantei fiecarei coloane.

**Exercitiul 7.1 - Feature importance**

Inainte sa rulezi, ordoneaza mental coloanele de la cea mai utila la cea mai putin utila. Apoi compara intuitia ta cu graficul.


In [ ]:
importante = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()

importante.plot(kind="barh")
plt.title("Ce coloane foloseste Random Forest?")
plt.xlabel("importanta")
plt.tight_layout()
plt.show()

importante.sort_values(ascending=False).round(3)


**Exercitiul 7.2 - Ce se intampla fara `cibil_score`?**

Antrenam acelasi model fara cea mai intuitiva coloana. Compara scorul cu modelul complet si cu baseline-ul.


In [ ]:
X_train_fara = X_train.drop(columns=["cibil_score"])
X_test_fara = X_test.drop(columns=["cibil_score"])

rf_fara = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_fara.fit(X_train_fara, y_train)

print(f"Random Forest complet: {rf.score(X_test, y_test):.3f}")
print(f"Fara cibil_score:      {rf_fara.score(X_test_fara, y_test):.3f}")
print(f"Baseline:              {accuracy_score(y_test, baseline_pred):.3f}")


## 8. Pragul de decizie

`predict_proba()` produce probabilitati. `predict()` foloseste implicit pragul 0.5: daca probabilitatea de aprobare este cel putin 0.5, modelul prezice `Aprobat`.

Dar intr-o problema reala, pragul poate fi ales in functie de costuri.

**Exercitiul 8.1 - Alege pragul cu cost minim**

Modifica cele doua costuri si observa ce prag devine mai bun.


In [ ]:
COST_APROBARE_GRESITA = 5
COST_RESPINGERE_GRESITA = 1

proba_aprobare = rf.predict_proba(X_test)[:, 1]

randuri = []
for prag in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    pred = (proba_aprobare >= prag).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    cost = fp * COST_APROBARE_GRESITA + fn * COST_RESPINGERE_GRESITA
    randuri.append({
        "prag": prag,
        "aprobari_gresite_FP": fp,
        "respingeri_gresite_FN": fn,
        "cost": cost,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
    })

pd.DataFrame(randuri).round(3)


<details>
<summary><b>Observatie</b></summary>

Pragul nu apartine doar modelului, ci si problemei. Daca aprobarile gresite sunt foarte scumpe, putem ridica pragul. Daca respingerile gresite sunt mai grave, putem cobori pragul.
</details>


## Tema scurta: 2 exercitii pe credit

Tema trebuie sa fie scurta si verificabila. Poti porni de la notebook si sa adaugi raspunsurile sub aceasta sectiune.

### Tema 1 - Regula manuala

Gaseste o regula manuala cu una, doua sau trei conditii care obtine cel putin `0.90` accuracy pe test.

De predat:

- codul regulii;
- accuracy, precision, recall si F1;
- 2-3 propozitii: de ce ai ales pragurile acelea?

### Tema 2 - Random Forest mic

Antreneaza 3 variante de Random Forest schimband doar:

- `n_estimators`
- `max_depth`
- `min_samples_leaf`
- `max_features`

De predat:

- un tabel cu cele 3 variante si scorurile lor;
- matricea de confuzie pentru cea mai buna varianta;
- 2-3 propozitii: ce tip de greseala ti se pare mai important aici, fals pozitiv sau fals negativ?


In [ ]:
# Starter Tema 1
# Modifica pragurile pana obtii cel putin 0.90 accuracy.

tema1_pred = (
    (X_test["cibil_score"] >= 550)
).astype(int)

print("Tema 1 accuracy: ", round(accuracy_score(y_test, tema1_pred), 3))
print("Tema 1 precision:", round(precision_score(y_test, tema1_pred), 3))
print("Tema 1 recall:   ", round(recall_score(y_test, tema1_pred), 3))
print("Tema 1 F1:       ", round(f1_score(y_test, tema1_pred), 3))


In [ ]:
# Starter Tema 2
# Lista cu 3 configuratii diferite.

configuratii = [
    {"n_estimators": 50, "max_depth": 3, "min_samples_leaf": 1, "max_features": "sqrt"},
    {"n_estimators": 100, "max_depth": 5, "min_samples_leaf": 5, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 1, "max_features": "sqrt"},
]

rezultate = []
modele_tema = []
for cfg in configuratii:
    model = RandomForestClassifier(random_state=42, n_jobs=-1, **cfg)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    modele_tema.append(model)
    rezultate.append({
        **cfg,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC_AUC": roc_auc_score(y_test, proba),
    })

pd.DataFrame(rezultate).round(3)


## Recap

Am parcurs acelasi traseu ca intr-un proiect mic de ML:

1. am inteles datele si baseline-ul;
2. am scris o regula manuala;
3. am antrenat un arbore care gaseste singur praguri;
4. am discutat Gini, adancime si overfitting;
5. am comparat arborele cu Random Forest;
6. am interpretat erorile, importanta coloanelor si pragul de decizie.

Ideea de retinut: un arbore este o lista de intrebari cu praguri. Random Forest combina multi arbori pentru stabilitate, dar trebuie comparat mereu cu reguli si modele mai simple.
